# QAG Evaluation Notebook

This notebook runs the QAG service on a set of benchmark questions. It sends each question to QAG and collects the answers. Then it processes all the answers and saves them.

**What it does**

1. Loads the benchmark questions from the GDC-Variant-Testset in `Data/truth.csv.gz`. The file also gives the intent and the correct answer for each question.
2. Sends each question to QAG 3 times and averages the numeric answers. Failed requests are retried with an increasing wait between tries.
3. Sorts each question into one of three groups:
   - **Answered:** at least one run returned a number.
   - **Ambiguous:** QAG replied at least once, but never with a number.
   - **Failed:** every run failed, for example because of a network or server error.
4. Lets you retry the failed questions as many times as needed.
5. Processes all the collected answers and saves them as Excel files in the `outputs` folder. For each answered question, it adds the difference between the averaged answer and the correct answer.

**Before you start**

- The QAG service must be running. By default the notebook expects it at `http://localhost:8000`.
- The questions are in `Data/truth.csv.gz`, which is part of this repository.
- Check the settings in the "Setup" section: the model to use, how long to wait for a reply, and whether to run all questions or only the first 10 as a quick test.

**Outputs**

All results are saved in the `outputs` folder:

- `answered_raw.jsonl`, `ambiguous_raw.jsonl`, `failed_raw.jsonl`: the raw results of every query, one file per group. They keep the full reply from each run.
- `good_answers.xlsx`: the answered questions, with the difference between the averaged answer and the correct answer.
- `ambiguous_answers.xlsx`: the ambiguous questions, with the reply from each run.
- `failed_answers.xlsx`: the questions that still failed after all retries.

Note: new results are added to the end of the raw result files. To start a fresh run, delete these files first.

Note: an Excel cell can hold at most 32,767 characters. Longer replies are cut to fit in the Excel files. The raw `.jsonl` files always keep the full reply.

## Libraries

In [ ]:
# Install the libraries this notebook needs.
# Tested with Python 3.14.8, pandas 3.0.6, numpy 2.5.4, tqdm 4.70.1,
# openpyxl 3.1.5, and requests 2.34.2.
%pip install -q pandas numpy tqdm openpyxl requests

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import json
import time
from tqdm import tqdm


## Setup

Settings for this run. Change them here before you start.

In [ ]:
# Run all questions (True), or only the first 10 as a quick test (False)
USE_FULL_SET = True

# The QAG service and the model it uses. The model name must match an alias
# in the model-config.yaml file of the QAG agent.
QAG_URL = "http://localhost:8000/endpoint/invoke"
MODEL_NAME = "(On-Prem) GPT OSS 120B"

# How long to wait for one reply, in seconds, before trying again
TIMEOUT = 60

# Folders and the input file
INPUT_DIR = "./Data"
OUTPUT_DIR = "./outputs"
INPUT_FILE = os.path.join(INPUT_DIR, "truth.csv.gz")

# Raw results, one line per question. New results are added to the end.
ANSWERED_PATH = os.path.join(OUTPUT_DIR, "answered_raw.jsonl")
AMBIGUOUS_PATH = os.path.join(OUTPUT_DIR, "ambiguous_raw.jsonl")
FAILED_PATH = os.path.join(OUTPUT_DIR, "failed_raw.jsonl")

# Processed results
GOOD_ANSWERS_PATH = os.path.join(OUTPUT_DIR, "good_answers.xlsx")
AMBIGUOUS_ANSWERS_PATH = os.path.join(OUTPUT_DIR, "ambiguous_answers.xlsx")
FAILED_ANSWERS_PATH = os.path.join(OUTPUT_DIR, "failed_answers.xlsx")

In [ ]:
# Create the input and output folders, if needed
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Helper functions

In [ ]:
# Column order for the saved files
COLUMN_ORDER = [
    "questions",
    "intent",
    "truth",
    "answer_average",
    "difference",
    "difference_percent",
    "response_full_1", "response_value_1", "response_status_1",
    "response_full_2", "response_value_2", "response_status_2",
    "response_full_3", "response_value_3", "response_status_3"
]

In [ ]:
def query_qag(row, n_runs=3, max_retries=3, backoff_start=5, backoff_factor=2):
    """Sends one question to QAG several times and collects the answers.

    Each run sends the question once. If the request fails, for example
    because of a network error or a status code other than 200, it is tried
    again, up to max_retries tries in total. The wait between tries starts at
    backoff_start seconds and is multiplied by backoff_factor each time. If
    QAG replies with something that is not valid JSON, the run fails right
    away, with no more tries.

    Each run ends with one of three statuses:
    - numeric: QAG replied with a number.
    - ambiguous: QAG replied, but the answer is not a number.
    - failed: every try failed.

    The numeric answers from all runs are averaged into answer_average.

    The function uses the MODEL_NAME, QAG_URL, and TIMEOUT settings from
    the Setup section.

    Args:
        row: A row with the questions, intent, and truth columns.
        n_runs: The number of times to send the question.
        max_retries: The maximum number of tries per run.
        backoff_start: The wait after the first failed try, in seconds.
        backoff_factor: The number to multiply the wait by after each
            failed try.

    Returns:
        A tuple (answered, not_answered). If at least one run returned a
        number, answered is the result dict and not_answered is None. If
        not, answered is None and not_answered is the result dict. The
        caller decides how to sort a not-answered question, for example
        into ambiguous or failed. The result dict holds the question,
        intent, truth, answer_average, and three per-run lists:
        responses_full, responses_values, and responses_status.
    """
    question = row['questions']
    intent = row['intent']
    truth = row['truth']

    all_responses_json = []
    all_response_values = []
    responses_status = []

    for run_idx in range(n_runs):
        retries = 0
        backoff = backoff_start
        final_status = None
        while retries < max_retries:
            try:
                payload = {
                    "query": question,
                    "backend": MODEL_NAME,
                    "output_type": "percentage"
                }
                resp = requests.post(
                    QAG_URL,
                    headers={"Content-Type": "application/json"},
                    json=payload,
                    timeout=TIMEOUT
                )
                status_code = resp.status_code
                if status_code == 200:
                    try:
                        resp_json = resp.json()
                        answer_candidate = resp_json.get("response", {}).get("answer", None)
                        if answer_candidate:
                            try:
                                answer_val = float(answer_candidate.replace('%', ''))
                                final_status = ("numeric", answer_val, resp_json)
                                break  # Got a number, so stop trying
                            except Exception:
                                final_status = ("ambiguous", "ambiguous", resp_json)
                                break  # An answer that is not a number is still a reply, so stop trying
                        else:
                            final_status = ("ambiguous", "ambiguous", resp_json)
                            break
                    except Exception as e:
                        final_status = ("failed", "failed", {"error": str(e)})
                        break  # The reply is not valid JSON
                else:
                    final_status = ("failed", "failed", {"status_code": status_code, "text": resp.text})
                    retries += 1
                    if retries < max_retries:
                        time.sleep(backoff)
                        backoff *= backoff_factor
            except Exception as e:
                final_status = ("failed", "failed", {"error": str(e)})
                retries += 1
                if retries < max_retries:
                    time.sleep(backoff)
                    backoff *= backoff_factor
        # End of one run: record only the final status
        if final_status is not None:
            responses_status.append(final_status[0])
            all_response_values.append(final_status[1])
            all_responses_json.append(final_status[2])
        else:
            responses_status.append("failed")
            all_response_values.append("failed")
            all_responses_json.append({"error": "unknown failure"})

    numeric_values = [v for v in all_response_values if isinstance(v, float)]
    averaged_val = float(np.mean(numeric_values)) if numeric_values else None

    result_dict = {
        "questions": question,
        "intent": intent,
        "truth": truth,
        "responses_full": all_responses_json,
        "responses_values": all_response_values,
        "responses_status": responses_status,
        "answer_average": averaged_val
    }

    if numeric_values:  # Answered: at least one run returned a number
        return result_dict, None
    else:  # Not answered: no run returned a number
        return None, result_dict

In [ ]:
def flatten_run_columns(result_dict, n=3):
    """Splits the per-run lists in a result into one column per run.

    A result from query_qag stores the answers from all runs in three lists:
    responses_full, responses_values, and responses_status. This function
    removes those lists and adds three keys for each run instead, for
    example response_full_1, response_value_1, and response_status_1. This
    gives one flat row per question, which is easier to save and read as a
    table.

    If there are fewer than n runs, the missing columns are set to None.
    Runs after the first n are dropped.

    The input dict is changed in place.

    Args:
        result_dict: The result dict returned by query_qag.
        n: The number of runs to turn into columns. It should match
            the n_runs value used in query_qag.

    Returns:
        The same dict, with the three lists replaced by per-run columns.
    """
    responses_full = result_dict.pop("responses_full", [])
    responses_values = result_dict.pop("responses_values", [])
    responses_status = result_dict.pop("responses_status", [])
    for i in range(n):
        result_dict[f"response_full_{i+1}"]   = responses_full[i]   if i < len(responses_full)   else None
        result_dict[f"response_value_{i+1}"]  = responses_values[i] if i < len(responses_values) else None
        result_dict[f"response_status_{i+1}"] = responses_status[i] if i < len(responses_status) else None
    return result_dict

In [ ]:
def order_columns(df):
    """Puts the columns in COLUMN_ORDER first, in that order.

    All other columns go at the end, in their current order. Columns in
    COLUMN_ORDER that are not in the DataFrame are added as empty columns.

    Args:
        df: The DataFrame to reorder, for example the good, ambiguous, or
            failed answers.

    Returns:
        A new DataFrame with the same rows and the columns reordered.
    """
    final_columns = COLUMN_ORDER + [col for col in df.columns if col not in COLUMN_ORDER]
    return df.reindex(columns=final_columns)

## Load the input file

In [ ]:
# GDC-Variant-Testset
gvt = pd.read_csv(INPUT_FILE, index_col=0)

In [ ]:
# The first 10 questions, for a quick test run
gvt_subset = gvt.iloc[:10].copy()

## Run QAG

### Run QAG queries

In [ ]:
# All questions, or the first 10 for a quick test (USE_FULL_SET is set in Setup)
run_set = gvt if USE_FULL_SET else gvt_subset
print(f"Questions to run: {len(run_set)}")

In [ ]:
# Send each question to QAG and save the result to the file for its group
with open(ANSWERED_PATH, "a", encoding="utf-8") as ans_f, \
     open(AMBIGUOUS_PATH, "a", encoding="utf-8") as amb_f, \
     open(FAILED_PATH, "a", encoding="utf-8") as fail_f:

    for idx, row in tqdm(run_set.iterrows(), total=len(run_set)):
        answered_result, not_answered_result = query_qag(row, n_runs=3)
        if answered_result:
            ans_f.write(json.dumps(flatten_run_columns(answered_result), ensure_ascii=False) + "\n")
            ans_f.flush()
        elif not_answered_result:
            statuses = not_answered_result.get("responses_status", [])
            if all(s == "failed" for s in statuses):
                fail_f.write(json.dumps(flatten_run_columns(not_answered_result), ensure_ascii=False) + "\n")
                fail_f.flush()
            else:
                amb_f.write(json.dumps(flatten_run_columns(not_answered_result), ensure_ascii=False) + "\n")
                amb_f.flush()

### Retry failed questions

In [ ]:
# Run this cell as many times as needed.
# Each run sends the failed questions again and keeps only the ones that still fail.

failed_df = pd.read_json(FAILED_PATH, orient="records", lines=True)
answered_df = pd.read_json(ANSWERED_PATH, orient="records", lines=True) if os.path.exists(ANSWERED_PATH) else pd.DataFrame()
ambiguous_df = pd.read_json(AMBIGUOUS_PATH, orient="records", lines=True) if os.path.exists(AMBIGUOUS_PATH) else pd.DataFrame()

if not failed_df.empty:
    rows_to_drop = []
    still_failed_rows = []

    with open(ANSWERED_PATH, "a", encoding="utf-8") as ans_f, \
         open(AMBIGUOUS_PATH, "a", encoding="utf-8") as amb_f:

        for idx, row in tqdm(failed_df.iterrows(), total=len(failed_df)):
            answered_result, not_answered_result = query_qag(row, n_runs=3)
            if answered_result:
                ans_f.write(json.dumps(flatten_run_columns(answered_result), ensure_ascii=False) + "\n")
                ans_f.flush()
                rows_to_drop.append(idx)
                print(f"Recovered: {answered_result['questions']}")
            elif not_answered_result:
                statuses = not_answered_result.get("responses_status", [])
                if all(s == "failed" for s in statuses):
                    # The row is already flat. to_json() saves missing values as null,
                    # so the file stays valid JSON.
                    still_failed_rows.append(json.loads(row.to_json()))
                else:
                    amb_f.write(json.dumps(flatten_run_columns(not_answered_result), ensure_ascii=False) + "\n")
                    amb_f.flush()
                    rows_to_drop.append(idx)
                    print(f"Ambiguous: {not_answered_result['questions']}")

    # Replace failed_raw.jsonl with the questions that still fail
    with open(FAILED_PATH, "w", encoding="utf-8") as fail_f:
        for fr in still_failed_rows:
            fail_f.write(json.dumps(fr, ensure_ascii=False) + "\n")
        fail_f.flush()

    print(f"\nQuestions with a reply: {len(rows_to_drop)}")
    print(f"Questions that still fail: {len(still_failed_rows)}")

else:
    print("No failed questions remain to retry.")

### Process and save answers

In [ ]:
answered_df = pd.read_json(ANSWERED_PATH, orient="records", lines=True)
ambiguous_df = pd.read_json(AMBIGUOUS_PATH, orient="records", lines=True)
failed_df = pd.read_json(FAILED_PATH, orient="records", lines=True)

In [ ]:
# Add the difference from the truth for each answered question
good_answers = []

for idx, row in answered_df.iterrows():
    avg_answer = row['answer_average']
    truth = row['truth']
    diff = avg_answer - truth if truth is not None else None
    diff_pct = (diff / truth * 100) if (diff is not None and truth not in [None, 0]) else None
    good_answers.append({**row, "difference": diff, "difference_percent": diff_pct})

good_df = pd.DataFrame(good_answers)

In [ ]:
# Save each group of questions to its own Excel file
for df, path in [(good_df, GOOD_ANSWERS_PATH),
                 (ambiguous_df, AMBIGUOUS_ANSWERS_PATH),
                 (failed_df, FAILED_ANSWERS_PATH)]:
    order_columns(df).to_excel(path, index=False, engine="openpyxl")
    print(f"Questions saved to {path}: {len(df)}")